# 📷 מחברת 3: זיהוי תווים אופטי (OCR) לטקסטים עבריים
## מבוא למדעי הרוח הדיגיטליים | אוניברסיטת אריאל | תשפ"ו
### פרופ' שי גורדין
---
OCR (Optical Character Recognition) = זיהוי תווים אופטי – טכנולוגיה שהופכת תמונות של טקסט לטקסט ניתן לעריכה ולחיפוש.
**נושאים:** (1) מהו OCR ולמה הוא חשוב? (2) Tesseract OCR – הכלי הפתוח המוביל (3) EasyOCR לעברית (4) הערכת איכות OCR (5) תיקון שגיאות (6) אתגרים מיוחדים בכתב יד עברי
**🎯 חשיבות לארכיאולוגיה ומדעי הרוח:** הגניזה הקהירית, כתבי יד ים המלח, תעודות עתיקות – כולם דורשים OCR לדיגיטציה.

---
## 🗺️ מדריך שימוש במחברת

| | |
|---|---|
| ⏱️ **זמן מוערך** | 🏫 בכיתה: ~30 דקות | 🏠 בבית: ~50 דקות (EasyOCR + Pipeline) |
| 💻 **היכן להריץ** | **Google Colab בלבד** — `!apt-get` לא עובד על מחשב רגיל |
| 🌐 **חיבור רשת** | נדרש להתקנות ולהורדת מודלים |
| ⚠️ **טרום-שיעור** | הריצו את **תא ההתקנה** בבית — EasyOCR מוריד ~2GB |

### 🏫 תכנית השיעור (30 דקות):
| זמן | פעילות |
|-----|--------|
| 0–3 דק׳ | התקנה + יבוא |
| 3–8 דק׳ | פונקציות עיבוד מוקדם |
| 8–17 דק׳ | Tesseract OCR לעברית |
| 17–28 דק׳ | **השוואת VLM — שלבים 1–2** (שיא השיעור!) |
| 28–30 דק׳ | דיון: מתי להשתמש במה? |

### 🏠 מה ממשיכים בבית:
- EasyOCR (Deep Learning)
- שלב 3 VLM: מסמך היסטורי מדומה
- הערכת איכות (CER, WER) + Pipeline מלא

### ❗ הפקודות `!apt-get` עובדות **רק ב-Colab**
בניגוד ל-`!pip install`, הן מתקינות תוכנות מערכת. **השתמשו ב-Colab!**

---


## חלק א: מהו OCR ולמה הוא חשוב?
OCR הוא תהליך שבו מחשב **"קורא"** טקסט מתמונה.
### תהליך OCR:
תמונה ← עיבוד מוקדם ← פילוח שורות/תיבות ← זיהוי תווים ← תיקון שגיאות ← טקסט
### דוגמאות לשימוש במדעי הרוח:
| מקור | אתגר | פתרון |
|------|-------|--------|
| עיתונות היסטורית (מעריב, הארץ) | דפוס ישן, פגמים | Tesseract + ניקוי |
| כתבי יד (גניזה קהירית) | כתב יד אישי | CNN + HTR |
| תעודות רשמיות עתיקות | קלף בלוי | שיפור תמונה + OCR |
| לוחות חרס (כתב יתד!) | צילום מיוחד | מודלים ייעודיים |
### מגבלות OCR לעברית:
- כיוון ימין-לשמאל (RTL) מסבך פילוח
- ניקוד (ּ ָ ִ ֵ) מסבך זיהוי
- אותיות דומות: ב/כ, ד/ר, ה/ח
- כתב יד מאוד שונה מדפוס
📖 לקריאה: Springmann, U. et al. (2018). *Ground Truth for Training OCR Engines on Historical Documents.* DH2018.

In [ ]:
# ════════════════════════════════════════════════════════════
# 🏠 משימת טרום-שיעור — הריצו תא זה בבית לפני השיעור
# ════════════════════════════════════════════════════════════
# הורדת מודל EasyOCR (~2GB) לוקחת זמן — עדיף לעשות בחיבור הביתי.
# אם כבר הרצתם פעם — המודל שמור בקאש ולא יורד שוב.

!pip install easyocr Pillow pytesseract google-generativeai python-bidi -q
!apt-get install -y tesseract-ocr tesseract-ocr-heb > /dev/null 2>&1
print("✅ ספריות OCR הותקנו!")
print("   • tesseract-ocr-heb      - Tesseract עם תמיכה בעברית")
print("   • easyocr                - OCR מבוסס Deep Learning (~2GB בהורדה ראשונה)")
print("   • google-generativeai    - ממשק ל-Gemini Vision (VLM)")
print("   • python-bidi            - תמיכה ב-RTL ביצירת תמונות")


In [ ]:
# יבוא ספריות
import pytesseract
from PIL import Image, ImageFilter, ImageEnhance, ImageOps
import easyocr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
from io import BytesIO
import os
import re
print("✅ ספריות יובאו!")

## חלק ב: עיבוד מוקדם של תמונות
### למה עיבוד מוקדם חשוב?
תמונות "גולמיות" לרוב אינן אידיאליות לOCR. עיבוד מוקדם משפר דרמטית את הדיוק:
| טכניקה | מטרה | מתי להשתמש |
|---------|-------|------------|
| **המרה לגווני אפור** | הפחתת מורכבות | תמיד |
| **ניגודיות** | הבלטת הטקסט | טקסט דהוי |
| **סף (Threshold)** | שחור-לבן בינארי | רקע לא אחיד |
| **הסרת רעש** | חלקות | טקסט עם "כתמים" |
| **ישור (Deskew)** | תיקון זווית | תמונות סרוקות |

In [ ]:
# ============================================================
# פונקציות עיבוד מוקדם של תמונות
# ============================================================

def preprocess_image(image, method='standard'):
    """
    עיבוד מוקדם של תמונה לשיפור דיוק OCR.
    
    פרמטרים:
        image  : PIL Image
        method : 'standard' / 'aggressive' / 'gentle'
    
    מחזיר: PIL Image מעובדת
    """
    # המרה לגווני אפור
    img = image.convert('L')
    
    if method == 'standard':
        # הגברת ניגודיות
        enhancer = ImageEnhance.Contrast(img)
        img = enhancer.enhance(2.0)
        # סף אדפטיבי
        img = img.point(lambda x: 0 if x < 128 else 255, '1')
        img = img.convert('L')
        
    elif method == 'aggressive':
        # חידוד קצוות
        img = img.filter(ImageFilter.SHARPEN)
        enhancer = ImageEnhance.Contrast(img)
        img = enhancer.enhance(3.0)
        img = img.point(lambda x: 0 if x < 100 else 255, '1')
        img = img.convert('L')
        
    elif method == 'gentle':
        # רק ניקוי קל
        img = img.filter(ImageFilter.MedianFilter(size=3))
        enhancer = ImageEnhance.Contrast(img)
        img = enhancer.enhance(1.5)
    
    return img


def show_preprocessing_comparison(image):
    """
    הצגת השוואה בין שיטות עיבוד מוקדם שונות.
    """
    methods = ['original', 'standard', 'aggressive', 'gentle']
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    for i, method in enumerate(methods):
        if method == 'original':
            img_show = image.convert('L')
            title = 'מקורית'
        else:
            img_show = preprocess_image(image, method=method)
            title = method
        
        axes[i].imshow(img_show, cmap='gray')
        axes[i].set_title(f'שיטה: {title}', fontsize=12, fontweight='bold')
        axes[i].axis('off')
    
    plt.suptitle('השוואת שיטות עיבוד מוקדם', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('01_preprocessing.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("💾 ההשוואה נשמרה: 01_preprocessing.png")


print("✅ פונקציות עיבוד מוקדם מוגדרות!")

In [ ]:
# ============================================================
# יצירת תמונת דוגמה עם טקסט עברי
# ============================================================
# נצור תמונה עם טקסט עברי לדמו

from PIL import Image, ImageDraw, ImageFont
import numpy as np

def create_sample_hebrew_image(text_lines, width=600, height=200, 
                                noise_level=10, font_size=24):
    """
    יצירת תמונת דוגמה עם טקסט עברי (לצורכי הדגמה).
    
    בדוגמה אמיתית, תשתמשו בתמונות סרוקות של מסמכים היסטוריים.
    """
    # יצירת תמונה לבנה
    img = Image.new('RGB', (width, height), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    
    # כתיבת הטקסט (בלי פונט עברי מיוחד – נשתמש בברירת מחדל)
    y_pos = 20
    for line in text_lines:
        draw.text((width - 20, y_pos), line, fill=(0, 0, 0), anchor='ra')
        y_pos += font_size + 8
    
    # הוספת רעש קל (מדמה סריקה)
    img_array = np.array(img)
    noise = np.random.randint(-noise_level, noise_level, img_array.shape)
    img_array = np.clip(img_array.astype(int) + noise, 0, 255).astype(np.uint8)
    
    return Image.fromarray(img_array)


# דוגמאות לטקסטים עבריים
sample_texts = [
    ["חפירות ארכיאולוגיות בארץ ישראל", 
     "מגלות שכבות היסטוריות רבות",
     "מהתקופה הכנענית ועד העות'מאנית"],
    
    ["הגניזה הקהירית",
     "אוסף של כ-300,000 קטעי מסמכים",
     "שנמצאו בבית הכנסת בן-עזרא, קהיר"],
    
    ["מדעי הרוח הדיגיטליים",
     "משלבים כלים חישוביים",
     "עם מחקר הומניסטי"]
]

# יצירת תמונות דוגמה
sample_images = []
for i, lines in enumerate(sample_texts):
    img = create_sample_hebrew_image(lines, noise_level=15)
    sample_images.append(img)
    img.save(f'sample_hebrew_{i+1}.png')
    print(f"✅ תמונת דוגמה {i+1} נוצרה")

# הצגת התמונות
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titles = ['📜 ארכיאולוגיה', '📜 גניזה קהירית', '💻 DH']
for i, (img, title) in enumerate(zip(sample_images, titles)):
    axes[i].imshow(img)
    axes[i].set_title(title, fontsize=12, fontweight='bold')
    axes[i].axis('off')
plt.suptitle('תמונות דוגמה לOCR', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("\n💡 בפרויקט אמיתי: תשתמשו בתמונות סרוקות של מסמכים היסטוריים!")

## חלק ג: Tesseract OCR
### מהו Tesseract?
**Tesseract** הוא מנוע OCR קוד-פתוח פותח על ידי Google, ונחשב לאחד הטובים בעולם למסמכים מודפסים.
**יתרונות:** חינמי, תומך ב-100+ שפות כולל עברית, קל לשימוש  
**חסרונות:** פחות טוב לכתב יד, רגיש לאיכות התמונה
### הפרמטרים החשובים:
- `lang='heb'` – שפה עברית
- `config='--psm 6'` – מצב פילוח (6 = בלוק טקסט אחיד)
- PSM modes: 3=automatic, 6=single block, 11=single word per line

In [ ]:
# ============================================================
# Tesseract OCR לעברית
# ============================================================

def run_tesseract(image, lang='heb', psm=6):
    """
    הרצת Tesseract OCR על תמונה.
    
    פרמטרים:
        image : PIL Image
        lang  : שפה ('heb', 'eng', 'heb+eng')
        psm   : Page Segmentation Mode (3=אוטומטי, 6=בלוק אחיד)
    
    מחזיר: הטקסט שזוהה
    """
    config = f'--psm {psm} --oem 3'
    text = pytesseract.image_to_string(
        image,
        lang=lang,
        config=config
    )
    return text.strip()


def tesseract_with_details(image, lang='heb'):
    """
    Tesseract עם מידע מפורט על כל תיבת זיהוי.
    מחזיר DataFrame עם מיקום ורמת ביטחון לכל מילה.
    """
    import pandas as pd
    data = pytesseract.image_to_data(
        image, lang=lang,
        output_type=pytesseract.Output.DATAFRAME
    )
    # סינון תוצאות עם ביטחון מספיק
    data = data[data['conf'] > 0].copy()
    return data


# ── הרצת Tesseract על הדוגמאות ──
print("🔍 מריץ Tesseract OCR על תמונות הדוגמה...")
print("=" * 55)

for i, img in enumerate(sample_images[:2]):  # 2 ראשונות
    print(f"\n📷 תמונה {i+1}:")
    print("-" * 40)
    
    # ניסיון עם תמונה מקורית
    text_raw = run_tesseract(img, lang='heb')
    print(f"  מקורית: {repr(text_raw[:100])}")
    
    # ניסיון עם עיבוד מוקדם
    img_processed = preprocess_image(img, method='standard')
    text_processed = run_tesseract(img_processed, lang='heb')
    print(f"  מעובדת: {repr(text_processed[:100])}")

print("\n💡 שים לב: התמונות נוצרו ללא פונט עברי ייעודי – לכן OCR עלול להיות לא מדויק.")
print("   בפרויקט אמיתי עם תמונות סרוקות, הדיוק גבוה הרבה יותר.")

## חלק ד: EasyOCR – OCR מבוסס Deep Learning
### מהו EasyOCR?
**EasyOCR** הוא כלי OCR מודרני המבוסס על רשתות נוירונים עמוקות (Deep Learning).
**יתרונות לעברית:**
- עדיף על Tesseract **לכתב יד**
- מטפל טוב יותר בפונטים מגוונים
- מחזיר רמת ביטחון (confidence) לכל מילה
- אין צורך בעיבוד מוקדם מורכב
**חסרונות:** איטי יותר, דורש GPU לביצועים מיטביים

In [ ]:
# ============================================================
# הגדרה: האם להפעיל EasyOCR?
# ============================================================
# EasyOCR מוריד מודל של ~2 ג'יגה-בייט בהפעלה הראשונה.
# בחיבור אינטרנט של 10 Mbps — זה לוקח כ-25 דקות.
#
# True  = דלגו על EasyOCR (ברירת מחדל בכיתה — נתמקד ב-Tesseract)
# False = הריצו EasyOCR (לשימוש בבית אחרי הורדת המודל מראש)

SKIP_EASYOCR = True   # ← שנו ל-False בבית להרצת EasyOCR

if SKIP_EASYOCR:
    print("🟡 EasyOCR מושבת — בכיתה נתמקד ב-Tesseract")
    print("   🏠 הריצו בבית עם SKIP_EASYOCR = False אחרי הורדת המודל מראש")
else:
    print("🟢 EasyOCR יופעל — ההורדה הראשונה עשויה לקחת כמה דקות")
    print("   (אם זה ה-Colab הראשון שלכם, המודל כבר שמור בקאש)")


In [ ]:
# ============================================================
# EasyOCR – OCR מבוסס Deep Learning
# ============================================================
# ⏳ ההפעלה הראשונה מורידה ~2 GB — המתינו בסבלנות

if SKIP_EASYOCR:
    print("🟡 EasyOCR מושבת — דלגנו")
    print("   כדי להפעיל, שנו SKIP_EASYOCR=False בתא למעלה")
    reader = None   # placeholder
else:
    print("🤖 מאתחל EasyOCR...")
    print("   (הורדה ראשונה = כ-2GB, עשוי לקחת מספר דקות)")
    print("   בסשן Colab חדש המודל מוריד כל פעם — זה נורמלי!")
    reader = easyocr.Reader(['he', 'en'], gpu=False)
    print("✅ EasyOCR מוכן!")


def run_easyocr(image_path_or_array, reader):
    """
    הרצת EasyOCR על תמונה.
    מחזיר: רשימת tuple (bounding_box, text, confidence)
    """
    if reader is None:
        print("⚠️ EasyOCR מושבת. שנו SKIP_EASYOCR=False ונסו שוב.")
        return []
    results = reader.readtext(image_path_or_array)
    return results


def display_easyocr_results(image, results, title='EasyOCR'):
    """
    הצגת תוצאות EasyOCR עם bounding boxes על התמונה.
    """
    import numpy as np
    img_arr = np.array(image)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # ציור bounding boxes
    img_with_boxes = img_arr.copy()
    for (bbox, text, conf) in results:
        pts = [(int(p[0]), int(p[1])) for p in bbox]
        # ציור מלבן
        import cv2  # noqa
        cv2.rectangle(img_with_boxes, pts[0], pts[2], (0, 255, 0), 2)
    
    ax1.imshow(img_arr)
    ax1.set_title('תמונה מקורית', fontsize=14)
    ax1.axis('off')
    
    ax2.imshow(img_with_boxes)
    ax2.set_title(f'{title} — תוצאות', fontsize=14)
    ax2.axis('off')
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n📝 טקסט שזוהה:")
    for i, (bbox, text, conf) in enumerate(results):
        print(f"  {i+1}. {text:30s}  ביטחון: {conf:.1%}")


---

## חלק ה׳: VLM לOCR — מודלי שפה-ראייה

> 📖 **קראו לפני השיעור** — ההסבר התיאורטי
> 🏫 **בכיתה** — נריץ יחד **שלבים 1–2** (כ-11 דקות)
> 🏠 **בבית** — שלב 3: מסמך היסטורי + שאילה מובנית

### מהם VLM?
**VLM** (Vision-Language Models) = מודלי AI המבינים **גם תמונות וגם שפה** — בו-זמנית.

בניגוד ל-Tesseract ו-EasyOCR שסורקים **פיקסל-פיקסל**, VLM **מבינים הקשר**:
- *"המילה כאן עמומה, אבל ההקשר מצביע על..."*
- *"זה עיתון, לכן הכותרת הראשית כנראה..."*
- *"שגיאת OCR כאן — אין מילה כזו בעברית"*

### השוואה:

| יכולת | Tesseract | EasyOCR | Gemini Vision |
|-------|:---------:|:-------:|:-------------:|
| עברית מודרנית ברורה | ✅ | ✅ | ✅ |
| ניקוד + כתיב חסר | ⚠️ | ⚠️ | ✅ |
| סריקה עם רעש/פגמים | ❌ | ⚠️ | ✅ |
| עיתונות היסטורית | ❌ | ⚠️ | ✅ |
| כתב יד | ❌ | ⚠️ | ✅ |
| OCR + ניתוח מסמך יחד | ❌ | ❌ | ✅ |
| עלות | חינם | חינם | חינם (tier בסיסי) |
| מהירות | ⚡ מהיר | 🐢 איטי | ⏱️ בינוני |

### איך עובד Gemini Vision?
```
תמונה + שאלה בעברית → [Gemini 1.5 Flash] → תשובה עם הבנת הקשר
```
מפתח API **חינמי** מ: https://aistudio.google.com/


In [ ]:
# ============================================================
# 🔧 הגדרת VLM + פונקציות עזר
# ============================================================

# ─── שלב 1: הכניסו מפתח מ-https://aistudio.google.com/ ─────
GEMINI_API_KEY = "YOUR_API_KEY_HERE"   # ← הדביקו כאן

# ─── ייבוא ──────────────────────────────────────────────────
import google.generativeai as genai
import re
from bidi.algorithm import get_display
from PIL import ImageFont, ImageDraw, ImageEnhance

# ─── הורדת פונט עברי ────────────────────────────────────────
import subprocess
subprocess.run(["wget", "-q",
    "https://github.com/alefalefalef/Alef/raw/master/Alef-Regular.ttf",
    "-O", "/tmp/AlefHebrew.ttf"], capture_output=True)

def get_hfont(size):
    try:
        return ImageFont.truetype('/tmp/AlefHebrew.ttf', size)
    except:
        return ImageFont.load_default()

# ─── יצירת תמונות בדיקה ─────────────────────────────────────
def make_ocr_image(lines, width=660, height=260, degrade=0, aged=False):
    """
    lines: רשימה של (טקסט, גודל_פונט)
    degrade: 0=נקי | 1=בינוני | 2=חמור
    aged: True = אפקט נייר ישן
    """
    bg = (245, 235, 195) if aged else (255, 255, 255)
    img = Image.new('RGB', (width, height), color=bg)
    draw = ImageDraw.Draw(img)
    y = 18
    for text, size in lines:
        font = get_hfont(size)
        visual = get_display(text)        # RTL תיקון עבור PIL
        bbox = draw.textbbox((0, 0), visual, font=font)
        x = (width - (bbox[2] - bbox[0])) // 2
        draw.text((x, y), visual, font=font,
                  fill=(25, 15, 8) if aged else (0, 0, 0))
        y += size + 14
    if degrade >= 1:
        arr = np.array(img).astype(float)
        arr += np.random.normal(0, 20 * degrade, arr.shape)
        img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
        img = img.filter(ImageFilter.GaussianBlur(radius=0.9 * degrade))
    if degrade >= 2:
        img = ImageEnhance.Contrast(img).enhance(0.55)
    if aged:
        arr = np.array(img).astype(float)
        # כתמי בלאי בשוליים
        margin = 18
        for edge_slice in [
            (slice(0, margin), slice(None)),
            (slice(-margin, None), slice(None)),
            (slice(None), slice(0, margin)),
            (slice(None), slice(-margin, None)),
        ]:
            patch = arr[edge_slice]
            arr[edge_slice] = patch * (0.82 + np.random.rand(*patch.shape) * 0.12)
        img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    return img

# ─── OCR ────────────────────────────────────────────────────
def run_tesseract(img):
    return pytesseract.image_to_string(img, lang='heb+eng',
                                        config='--psm 6').strip()

def run_gemini(img, prompt="קרא את הטקסט בתמונה במדויק. החזר רק את הטקסט ללא הסברים."):
    if not GEMINI_READY:
        return "[הגדירו GEMINI_API_KEY כדי לראות תוצאה]"
    return genai.GenerativeModel('gemini-1.5-flash').generate_content(
        [prompt, img]).text.strip()

# ─── השוואה חזותית ──────────────────────────────────────────
def compute_cer(ref, hyp):
    """Character Error Rate — מדד דיוק OCR (0=מושלם, 1=גרוע)."""
    ref = re.sub(r'\s+', ' ', ref).strip()
    hyp = re.sub(r'\s+', ' ', hyp).strip()
    m, n = len(ref), len(hyp)
    if m == 0: return 0.0
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp.copy(), i
        for j in range(1, n + 1):
            dp[j] = prev[j-1] if ref[i-1] == hyp[j-1] else 1 + min(prev[j], dp[j-1], prev[j-1])
    return dp[n] / m

def show_ocr_comparison(title, img, tess_out, vlm_out, ref=None):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    axes[0].imshow(img); axes[0].set_title('📸 תמונת הקלט', fontsize=11); axes[0].axis('off')

    for ax, text, color, label in [
        (axes[1], tess_out or '(אין פלט)', '#FFF0F0', '⚙️ Tesseract (קלאסי)'),
        (axes[2], vlm_out,                '#F0FFF0', '🧠 Gemini Vision (VLM)'),
    ]:
        ax.text(0.05, 0.96, text, transform=ax.transAxes,
                fontsize=9, va='top', ha='left', family='monospace',
                bbox=dict(boxstyle='round,pad=0.6', facecolor=color, alpha=0.95))
        ax.set_title(label, fontsize=11); ax.axis('off')

    plt.tight_layout(); plt.show()

    if ref:
        ct = compute_cer(ref, tess_out)
        cg = compute_cer(ref, vlm_out)
        print(f"\n📊 שגיאות תווים (CER):  Tesseract {ct:.0%}  →  Gemini {cg:.0%}")
        imp = (ct - cg) / max(ct, 0.001) * 100
        icon = '✅' if imp > 10 else ('➡️' if imp >= 0 else '⚠️')
        print(f"   {icon} שיפור: {imp:+.0f}%")

# ─── הגדרת Gemini ───────────────────────────────────────────
GEMINI_READY = False
if GEMINI_API_KEY != "YOUR_API_KEY_HERE":
    genai.configure(api_key=GEMINI_API_KEY)
    GEMINI_READY = True
    print("✅ Gemini Vision מוכן!")
else:
    print("⚠️  Gemini לא מוגדר — תוצאות VLM יוצגו כ-placeholder")
    print("   קבלו מפתח חינמי: https://aistudio.google.com/")
print("✅ כל הכלים מוכנים!")


In [ ]:
# ============================================================
# 🏫 שלב 1: טקסט ברור — נקודת ייחוס
# ============================================================
# שאלה: כשהתמונה נקייה, האם בכלל יש הבדל בין Tesseract ל-VLM?

LINES_SAMPLE = [
    ("הספרייה הלאומית של ישראל", 32),
    ("אוסף עיתונות היסטורית דיגיטלית", 24),
    ("Jerusalem, 15 April 1940", 20),
    ("כ״ז בניסן, תש״פ", 20),
]
REF_TEXT = "הספרייה הלאומית של ישראל\nאוסף עיתונות היסטורית דיגיטלית\nJerusalem, 15 April 1940\nכ״ז בניסן, תש״פ"

img1 = make_ocr_image(LINES_SAMPLE, degrade=0)
tess1 = run_tesseract(img1)
vlm1  = run_gemini(img1)

show_ocr_comparison("שלב 1 — טקסט ברור: שניהם אמורים לעבוד", img1, tess1, vlm1, REF_TEXT)
print("\n💡 תצפית: על טקסט נקי שניהם עובדים — Tesseract מהיר ובחינם, Gemini מדויק יותר.")
print("   שאלה לדיון: מה יקרה כשהתמונה פחות נקייה?")


In [ ]:
# ============================================================
# 🏫 שלב 2: סריקה עם רעש ופגמים — כישלון Tesseract
# ============================================================
# מדמה סריקה ישנה / רזולוציה נמוכה / נייר פגום.
# זהו הסיטואציה הנפוצה ביותר במחקר היסטורי!

img2 = make_ocr_image(LINES_SAMPLE, degrade=2)  # אותו טקסט, עם רעש חמור
tess2 = run_tesseract(img2)
vlm2  = run_gemini(img2)

show_ocr_comparison("שלב 2 — סריקה עם רעש: כאן מתגלה הפער", img2, tess2, vlm2, REF_TEXT)

print("\n💡 מה קרה כאן?")
print("   Tesseract: מנסה לפענח פיקסל-פיקסל — הרעש מטעה אותו לחלוטין.")
print("   Gemini:    מבין שזה טקסט ספרייה/עיתון ומשחזר את המשמעות גם מרעש.")
print()
print("🔬 עיקרון: VLM אינו רק קורא פיקסלים — הוא *מבין* מה כתוב.")
print("   לכן הוא עמיד בפני דעיכה, כתמים, ופונטים לא מוכרים.")


In [ ]:
# ============================================================
# 🏠 שלב 3: מסמך היסטורי + שאילה מובנית (בבית)
# ============================================================
# יתרון ייחודי של VLM: OCR + ניתוח מסמך — בשאילה אחת.
# זה בדיוק מה שחוקרי DH צריכים לעיתונות היסטורית!

LINES_HIST = [
    ("דבר — יומון הפועלים", 30),
    ("יום שלישי, ח׳ בתמוז תש״א", 20),
    ("כינוס הסתדרות הכללית בתל-אביב", 22),
    ("The General Federation of Labour", 16),
    ("שנה ט״ז, גיליון מס׳ 4,702", 16),
]
REF_HIST = "דבר — יומון הפועלים\nיום שלישי, ח׳ בתמוז תש״א\nכינוס הסתדרות הכללית בתל-אביב\nThe General Federation of Labour\nשנה ט״ז, גיליון מס׳ 4,702"

img3 = make_ocr_image(LINES_HIST, degrade=1, aged=True)
tess3 = run_tesseract(img3)

# ── שאילה מובנית: OCR + ניתוח מסמך בשאילה אחת ──────────────
structured_prompt = """קרא את המסמך בתמונה והחזר בפורמט הבא:
**טקסט מלא:**
[הטקסט שרואים]

**ניתוח:**
- סוג מסמך: (עיתון / ספר / מכתב / אחר)
- שם הפרסום:
- תאריך:
- שפות:
- כותרת ראשית:"""

vlm3 = run_gemini(img3, prompt=structured_prompt)

show_ocr_comparison("שלב 3 — מסמך היסטורי: OCR + ניתוח מובנה", img3, tess3, vlm3, REF_HIST)

print("\n🎯 הכוח האמיתי של VLM למחקר היסטורי:")
print("   שאילה אחת = קריאת הטקסט + זיהוי הפרסום + חילוץ תאריך + סיווג סוג מסמך.")
print("   לעומת Tesseract שמחזיר גוש טקסט (לעתים שגוי) ללא מטא-נתונים.")
print()
print("💬 שאלות לדיון:")
print("   1. מה עלות השימוש ב-Gemini API לדיגיטציה של 10,000 עמודים?")
print("   2. מתי בכל זאת נעדיף Tesseract (עלות? פרטיות? מהירות?)")
print("   3. איך תשלבו VLM-OCR בפרויקט הוויקיפדיה שלכם?")


## חלק ה: הערכת איכות OCR
### מדדי איכות:
| מדד | הסבר | נוסחה |
|-----|-------|-------|
| **CER** (Character Error Rate) | שיעור שגיאות ברמת תווים | (הכנסות + מחיקות + החלפות) / סה"כ תווים |
| **WER** (Word Error Rate) | שיעור שגיאות ברמת מילים | (שגיאות מילים) / סה"כ מילים |
### Ground Truth:
כדי לחשב CER/WER, צריך **Ground Truth** – הטקסט הנכון לידנו.  
בפרויקטי OCR, יוצרים Ground Truth על ידי הקלדה ידנית של חלק מהמסמכים.

In [ ]:
# ============================================================
# חישוב מדדי איכות OCR: CER ו-WER
# ============================================================

def levenshtein_distance(s1, s2):
    """
    חישוב מרחק לוונשטיין (מספר פעולות עריכה מינימלי בין שני מחרוזות).
    בסיס לחישוב CER ו-WER.
    """
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    
    if len(s2) == 0:
        return len(s1)
    
    prev_row = range(len(s2) + 1)
    
    for i, c1 in enumerate(s1):
        curr_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions  = prev_row[j + 1] + 1
            deletions   = curr_row[j] + 1
            substitutions = prev_row[j] + (c1 != c2)
            curr_row.append(min(insertions, deletions, substitutions))
        prev_row = curr_row
    
    return prev_row[-1]


def compute_cer(ground_truth, ocr_output):
    """
    חישוב CER – Character Error Rate.
    0 = מושלם, 1 = 100% שגיאות
    """
    gt_clean  = re.sub(r'\s+', '', ground_truth.strip())
    ocr_clean = re.sub(r'\s+', '', ocr_output.strip())
    
    if len(gt_clean) == 0:
        return 0.0
    
    dist = levenshtein_distance(gt_clean, ocr_clean)
    return dist / len(gt_clean)


def compute_wer(ground_truth, ocr_output):
    """
    חישוב WER – Word Error Rate.
    """
    gt_words  = ground_truth.strip().split()
    ocr_words = ocr_output.strip().split()
    
    if len(gt_words) == 0:
        return 0.0
    
    dist = levenshtein_distance(gt_words, ocr_words)
    return dist / len(gt_words)


# ── הדגמה ──
print("📊 הדגמת חישוב CER/WER:")
print("=" * 55)

# דוגמאות השוואה
examples = [
    ("חפירות ארכיאולוגיות", "חפירות ארכאולוגיות"),  # שגיאת כתיב קטנה
    ("הגניזה הקהירית",      "הגניזה קהירית"),        # מילה חסרה
    ("מדעי הרוח הדיגיטליים", "מדעי הרוח הדיגיטל1ים"),  # שגיאת OCR טיפוסית
]

for gt, ocr in examples:
    cer = compute_cer(gt, ocr)
    wer = compute_wer(gt, ocr)
    print(f"\n  GT : {gt}")
    print(f"  OCR: {ocr}")
    print(f"  CER={cer:.3f} ({cer*100:.1f}%)  |  WER={wer:.3f} ({wer*100:.1f}%)")

print("\n💡 CER < 5% = OCR איכותי מאוד")
print("   CER < 15% = OCR סביר (ניתן לתיקון)")
print("   CER > 30% = OCR גרוע (דורש שיפור)")

## חלק ו: תיקון שגיאות OCR
### שיטות תיקון:
1. **תיקון ידני**: הכי מדויק אבל איטי
2. **חיפוש במילון**: בדיקת כל מילה מול מילון עברי
3. **מודלי שפה**: GPT/Claude לתיקון הקשרי
4. **כלים ייעודיים**: [Transkribus](https://readcoop.eu/transkribus/), [eScriptorium](https://escriptorium.fr/)

### Transkribus – כלי מוביל לכתב יד:
Transkribus מיועד לכתבי יד היסטוריים ומשתמש בלמידה עמוקה.  
מחקרי גניזה קהירית משתמשים בו נרחבות.

In [ ]:
# ============================================================
# תיקון שגיאות OCR
# ============================================================

# מילון תיקונים נפוצים לעברית (OCR errors)
COMMON_OCR_CORRECTIONS = {
    # אותיות דומות
    'ה0': 'הו',    # ה + 0 במקום ו
    'ר1': 'רי',    # ר + 1 במקום י
    '0': 'ו',      # 0 במקום ו (נפוץ מאוד!)
    '1': 'י',      # 1 במקום י
    
    # מילים נפוצות שנגרסות
    'ישרא1ל': 'ישראל',
    'ירוש1ים': 'ירושלים',
    'ארכא0ל0גיה': 'ארכיאולוגיה',
}


def basic_ocr_correction(text, corrections=None):
    """
    תיקון בסיסי של שגיאות OCR נפוצות.
    
    פרמטרים:
        text        : טקסט פלט OCR לתיקון
        corrections : מילון תיקונים {שגוי: נכון}
    
    מחזיר: טקסט מתוקן
    """
    if corrections is None:
        corrections = COMMON_OCR_CORRECTIONS
    
    corrected = text
    for wrong, right in corrections.items():
        corrected = corrected.replace(wrong, right)
    
    return corrected


def spell_check_hebrew(text, word_list=None):
    """
    בדיקת איות בסיסית לעברית.
    משווה מול רשימת מילים נפוצות.
    
    הערה: לתיקון מקצועי, מומלץ להשתמש ב-hunspell-he.
    """
    if word_list is None:
        # מדגם של מילים נפוצות בתחום
        word_list = set([
            'ארכיאולוגיה', 'חפירה', 'ממצאים', 'שכבה', 'אתר',
            'היסטוריה', 'תקופה', 'ממלכה', 'שושלת', 'מלחמה',
            'דיגיטלי', 'מחשב', 'נתונים', 'אלגוריתם', 'קורפוס',
            'ירושלים', 'תל', 'חרס', 'ברונזה', 'ברזל',
            'ישראל', 'יהודה', 'שומרון', 'גליל', 'נגב'
        ])
    
    words = text.split()
    unknown = []
    for word in words:
        clean = re.sub(r'[^\u05D0-\u05EA]', '', word)
        if len(clean) > 2 and clean not in word_list:
            unknown.append(clean)
    
    return unknown


# ── הדגמה ──
print("🔧 הדגמת תיקון שגיאות OCR:")
print("=" * 55)

test_texts = [
    "ה0א נמצא בחפיר0ת ב1ר0של1ם",
    "ממצא1 ארכא0ל0ג1ים חשוב1ם",
    "מדע1 הר0ח הד1ג1טל1ים"
]

for text in test_texts:
    corrected = basic_ocr_correction(text)
    print(f"\n  שגוי:    {text}")
    print(f"  מתוקן:   {corrected}")

print("\n💡 לתיקון מקצועי: ")
print("   pip install pyenchant  # בדיקת איות")
print("   או שימוש ב-Claude API לתיקון הקשרי!")

## חלק ז: Pipeline מלא – מתמונה לטקסט ניתן לחיפוש
### תהליך מלא לדיגיטציה:
1. **סריקה** → 300+ DPI, אפור/צבע
2. **עיבוד מוקדם** → ניגודיות, ישור, חיתוך
3. **OCR** → Tesseract / EasyOCR
4. **הערכה** → CER/WER מול Ground Truth
5. **תיקון** → ידני / אוטומטי
6. **אינדקס** → חיפוש פולטקסט

In [ ]:
# ============================================================
# Pipeline מלא – מתמונה לטקסט
# ============================================================

def full_ocr_pipeline(image, lang='heb', method='standard'):
    """
    Pipeline מלא של OCR לעברית.
    
    שלבים:
    1. עיבוד מוקדם
    2. Tesseract OCR
    3. תיקון בסיסי
    
    מחזיר: dict עם כל שלבי העיבוד
    """
    results = {
        'original_image': image,
        'steps': {}
    }
    
    # שלב 1: עיבוד מוקדם
    preprocessed = preprocess_image(image, method=method)
    results['steps']['preprocessing'] = preprocessed
    
    # שלב 2: OCR עם שיטות שונות
    text_raw       = run_tesseract(image, lang=lang)
    text_processed = run_tesseract(preprocessed, lang=lang)
    results['steps']['tesseract_raw']       = text_raw
    results['steps']['tesseract_processed'] = text_processed
    
    # שלב 3: תיקון
    text_corrected = basic_ocr_correction(text_processed)
    results['steps']['corrected'] = text_corrected
    
    # שלב 4: מידע נוסף
    results['word_count'] = len(text_corrected.split())
    results['char_count'] = len(re.sub(r'\s', '', text_corrected))
    
    return results


# ── הרצת ה-Pipeline ──
print("🔄 מריץ Pipeline OCR מלא...")
print("=" * 55)

for i, img in enumerate(sample_images):
    print(f"\n📷 מסמך {i+1}:")
    result = full_ocr_pipeline(img)
    
    print(f"  Tesseract (גולמי)  : {repr(result['steps']['tesseract_raw'][:60])}")
    print(f"  Tesseract (מעובד)  : {repr(result['steps']['tesseract_processed'][:60])}")
    print(f"  לאחר תיקון         : {repr(result['steps']['corrected'][:60])}")
    print(f"  מילים: {result['word_count']} | תווים: {result['char_count']}")

print("\n✅ Pipeline הושלם!")

## 📝 תרגילים

### תרגיל 1 – בסיסי ⭐
הורידו תמונה של עיתון עברי ישן מ[National Library of Israel](https://www.nli.org.il/) והריצו עליה OCR.  
השוו: Tesseract לעומת EasyOCR – מי מדויק יותר?

### תרגיל 2 – בינוני ⭐⭐
צרו **Ground Truth** קטן (5-10 שורות) והשוו את דיוק Tesseract לפני ואחרי עיבוד מוקדם.  
חשבו CER ו-WER לכל שיטה.

### תרגיל 3 – מתקדם ⭐⭐⭐
השתמשו ב-**Claude API** לתיקון שגיאות OCR:
```python
# שלחו את טקסט ה-OCR ל-Claude עם הנחיה:
# "תקן את שגיאות ה-OCR בטקסט העברי הבא, מבלי לשנות תוכן:"
```

---
## 🔗 משאבים
- [Tesseract OCR Documentation](https://tesseract-ocr.github.io/)
- [EasyOCR GitHub](https://github.com/JaidedAI/EasyOCR)
- [Transkribus](https://readcoop.eu/transkribus/) – לכתב יד היסטורי
- [National Library of Israel – OCR](https://www.nli.org.il/)
- [Programming Historian: OCR with Tesseract](https://programminghistorian.org/en/lessons/working-with-text-files)

---
## 🏠 מטלת בית — OCR על טקסט אמיתי

**מה לעשות:**
1. בחרו **טקסט מודפס עברי** — עיתון ישן, ספר, או מסמך (שני עמודים לפחות)
2. **סרקו** באמצעות הסורק או אפליקציית הטלפון (Microsoft Lens / Google PhotoScan)
3. **העלו ל-Colab** את קובץ התמונה (גרירה לחלונית הקבצים בצד שמאל)
4. הריצו OCR עם `run_tesseract()` על התמונה שלכם
5. **בדקו** 10 שורות ידנית וסמנו שגיאות בצבע אחר

### העלאת קובץ ל-Colab:
```python
# לאחר העלאה, קראו את התמונה:
from PIL import Image
my_image = Image.open('שם_הקובץ.jpg')  # שנו לשם הקובץ שלכם
my_text = run_tesseract(my_image, lang='heb')
print(my_text)
```

**מה להגיש:**
- קובץ `OCR_output.txt` עם הטקסט שזוהה
- דוח קצר (טבלה): שורות שנבדקו, שגיאות שנמצאו, CER מוערך

**צריכים עזרה?** שאלו את [Gemini](https://gemini.google.com):  
> *"אני מנסה להריץ Tesseract OCR בפייתון על תמונה עברית. קיבלתי: [שגיאה]. כיצד לפתור?"*

### ✅ רשימת בדיקה לפני הגשה
- [ ] הרצתי OCR על תמונה אמיתית (לא על תמונת הדוגמה מהמחברת)
- [ ] בדקתי לפחות 10 שורות ידנית
- [ ] יש לי קובץ טקסט פלט
- [ ] כתבתי הערות על שגיאות שמצאתי
